In [42]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
load_dotenv()

True

In [55]:
llm = ChatOpenAI(model='gpt-4o-mini')

classify_prompt = ChatPromptTemplate.from_template(
    'Classify this email in ONE word (Billing/Technical/Spam/Other) and confidence score (0-1):\n'
    'Subject: {subject}\nBody: {body}'
)

parser = StrOutputParser()
clasify_chain = classify_prompt | llm | parser


In [56]:
emails = [
    {'subject': 'Payment failed',   'body': 'Charged twice this month.'},
    {'subject': 'App crash',        'body': 'Crashes on Settings page.'},
    {'subject': 'Add Slack?',       'body': 'Want Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',  'body': 'Click here NOW!!!'},
    {'subject': 'Invoice question', 'body': 'Can I get annual invoice?'},
]

output = clasify_chain.batch(emails,config={"batch_size": 5})

In [57]:
output

['Classification: Billing  \nConfidence Score: 0.9',
 'Classification: Technical  \nConfidence Score: 0.9',
 'Classification: Other  \nConfidence Score: 0.85',
 'Classification: Spam  \nConfidence Score: 0.95',
 'Classification: Billing  \nConfidence score: 0.9']

In [58]:
# outputs_from_llm = [
#     'CATEGORY: Billing\nURGENCY: High',        # normal
#     'CATEGORY:  Billing\nURGENCY: High',       # extra space!
#     'Category: Billing\nUrgency: High',         # different case!
#     'CATEGORY: Billing (payment issue)\nURGENCY: High',  # extra text!
#     'I think this is CATEGORY: Billing\nURGENCY: High',  # explanation first!
# ]

In [59]:
# pip install pydantic

In [60]:
from pydantic import BaseModel, Field
from typing import Literal

In [61]:
class Output(BaseModel):
    category: Literal['Billing', 'Technical', 'Spam', 'Other']
    confidence: float = Field(..., ge=0, le=1.0, description="Confidence score between 0 and 1")
   

In [62]:
Output(
    category='Billing',
    confidence=0.96
).category

'Billing'

In [63]:
custom_parser = PydanticOutputParser(pydantic_object=Output)

In [66]:
chain = classify_prompt | llm 

In [67]:
chain.invoke({'subject': 'Billing',
              'body': 'Charged twice this month.'})
# https://python.langchain.com/docs/concepts/lcel/

AIMessage(content='Classification: Billing  \nConfidence Score: 0.95', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 43, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b8c19d84b4', 'id': 'chatcmpl-EG41QQn3jKa5vGkRZNPsuyxO5Qyly', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02f26-047a-7fe2-aff8-c592f31e2e30-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 11, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})